# pr177 bug repros

In [1]:
import numpy as np
import torch
import matplotlib

matplotlib.use("Agg")

from quantem.core.datastructures.dataset2d import Dataset2d
from quantem.core.datastructures.dataset4dstem import Dataset4dstem
from quantem.diffraction.polar import PairDistributionFunction

torch.set_default_dtype(torch.float32)

In [2]:
# synthetic ring pattern shared across tests
n_pixels = 256
pixel_size = 0.05  # 1/A per pixel
center = (n_pixels - 1) / 2.0
y, x = np.ogrid[:n_pixels, :n_pixels]
r = np.sqrt((y - center) ** 2 + (x - center) ** 2)
k_2d = r * pixel_size
pattern = (
    500 * np.exp(-((k_2d - 2.0) ** 2) / (2 * 0.3 ** 2))
    + 200 * np.exp(-((k_2d - 4.0) ** 2) / (2 * 0.5 ** 2))
).astype(np.float32)

ds2d = Dataset2d.from_array(
    array=pattern, name="review_repro",
    origin=(0, 0), sampling=(pixel_size, pixel_size),
    units=["1/Angstrom", "1/Angstrom"],
)

array_4d = np.stack([[pattern, pattern], [pattern, pattern]])
ds4d = Dataset4dstem.from_array(
    array=array_4d, name="review_repro_4d",
    origin=(0, 0, 0, 0), sampling=(1, 1, pixel_size, pixel_size),
    units=["nm", "nm", "1/Angstrom", "1/Angstrom"],
)

## qq miscalibration with non-default radial params

`polar4dstem.py` stores `origin[3] = radial_min * pixel_size` (physical units)
but `qq` does `(arange(n) - origin_r) * sampling_r` treating it as a pixel offset.
Shifts the whole q-axis.

notes:
q-axis should start at 10 * 0.05 = 0.5 1/A and step by 2 * 0.05 = 0.1 1/A, but polar4dstem.py stores origin[3] as physical units while polar.py treats origin[3] as pixel index and then multiplying by sampling[3] again. shifts everything by 0.55 1/A, bin 0 goes negative (to -0.05 1/A) --> wrong q values?

In [3]:
def test1():
    radial_min, radial_step = 10.0, 2.0
    pdf = PairDistributionFunction.from_data(
        ds2d, find_origin=False,
        radial_min=radial_min, radial_step=radial_step,
    )
    qq = pdf.qq
    n_radial = len(qq)
    qq_expected = (radial_min + np.arange(n_radial) * radial_step) * pixel_size
    max_err = float(np.max(np.abs(qq - qq_expected)))
    assert int(np.sum(qq < 0)) == 0, f"{int(np.sum(qq < 0))} negative q values"
    assert max_err < 1e-6, f"qq off by {max_err:.4f} 1/A"

## stale `reduced_pdf_damped`

`calculate_Gr` sets `self.reduced_pdf_damped` when damping is on but never
clears it. The return path prefers the damped attr if it exists, so a
follow-up undamped call silently returns stale data.

notes: 
calculate_Gr with damp_origin_oscillations=True stores result in self.reduced_pdf_damped but when called again with damp_origin_oscillations=False, computes correct undamped G(r) internally into self._reduced_pdf. Since reduced_pdf_damped was set by the first call and never cleared, the second call returns the old damped result

In [4]:
def test2():
    pdf = PairDistributionFunction.from_data(ds2d, find_origin=False)
    pdf.calculate_Gr(
        k_min=0.3, k_max=5.5, r_min=0.0, r_max=15.0, r_step=0.02,
        damp_origin_oscillations=True, r_cut=1.0, returnval=True,
    )
    _, Gr_undamped = pdf.calculate_Gr(
        k_min=0.3, k_max=5.5, r_min=0.0, r_max=15.0, r_step=0.02,
        damp_origin_oscillations=False, returnval=True,
    )
    Gr_computed = pdf._to_numpy(pdf._reduced_pdf)
    contamination = float(np.max(np.abs(Gr_undamped - Gr_computed)))
    assert np.array_equal(Gr_undamped, Gr_computed), f"contamination {contamination:.1f}"

## `estimate_density` ignores k_max

Fit mask is `k >= kmin` with no upper bound — correction gets applied
to bins way past the user's `k_max`.

notes: there's no no & (k <= self.kmax), so when k_max = 3.0 is set, density correction is still applied to al lbins from 0.5 out to the edge of detector (approx ~6 1/A). 67 out of 118 bins in mask are above the user's k_max, 57% of the correction is happening outside the window that calculate_Gr actually uses

In [5]:
def test3():
    pdf = PairDistributionFunction.from_data(ds2d, find_origin=False)
    pdf.calculate_Gr(k_min=0.5, k_max=3.0)
    k = np.asarray(pdf.qq)
    n_bins_inside = int(np.sum((k >= pdf.kmin) & (k <= pdf.kmax)))
    n_bins_total = int(np.sum(k >= pdf.kmin))
    n_bins_outside = n_bins_total - n_bins_inside
    assert n_bins_outside == 0, f"{n_bins_outside}/{n_bins_total} bins outside [{pdf.kmin}, {pdf.kmax}]"

## empty mask -> silent NaN

All-False mask makes `calculate_radial_mean` average nothing → NaN
propagates through bg, F(k), sine transform

notes: (minor) if all-False real space mask is passed, calculate_radial_mean averages zero pixels -> NaN. should raise a ValueError

In [6]:
def test4():
    pdf = PairDistributionFunction.from_data(ds4d, find_origin=False)
    mask_empty = np.zeros((2, 2), dtype=bool)
    try:
        _, Gr = pdf.calculate_Gr(
            k_min=0.3, k_max=5.5, mask_realspace=mask_empty, returnval=True,
        )
    except (ValueError, RuntimeError):
        return
    n_nan = int(np.sum(np.isnan(Gr)))
    assert n_nan == 0, f"{n_nan}/{len(Gr)} NaN in G(r)"

In [7]:
def test(fn):
    try:
        fn()
        status = "PASS"
        detail = ""
    except AssertionError as exc:
        status = "FAIL"
        detail = str(exc)
    print(f"{fn.__name__}: {status} {('- ' + detail) if detail else ''}")

test(test1)
test(test2)
test(test3)
test(test4)

test1: FAIL - 1 negative q values
test2: FAIL - contamination 57.0
test3: FAIL - 67/118 bins outside [0.5, 3.0]
test4: FAIL - 999/1000 NaN in G(r)
